In [1]:
from datasets import load_dataset
import pandas as pd
from rag.models.embeddings import Embedder
from rag.models.tokenizer import Tokenizer
ds = load_dataset("k-mktr/russian_synodal_ru")

c:\Users\daavl\OneDrive\Документы\VScode\RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


In [2]:
df = pd.DataFrame(ds['train'])
df.head()

,translation_id,translation,language,book_number,book,book_abbr,chapter,verse,verse_number,verse_suffix,verse_id,text
0,russian_synodal_ru,Russian Synodal Bible (1876),Russian,13,1 Chronicles,1Chr,1,1,1,None,1Chr_1_1,"Адам, Сиф, Енос,"
1,russian_synodal_ru,Russian Synodal Bible (1876),Russian,13,1 Chronicles,1Chr,1,2,2,None,1Chr_1_2,"Каинан, Малелеил, Иаред,"
2,russian_synodal_ru,Russian Synodal Bible (1876),Russian,13,1 Chronicles,1Chr,1,3,3,None,1Chr_1_3,"Енох, Мафусал, Ламех,"
3,russian_synodal_ru,Russian Synodal Bible (1876),Russian,13,1 Chronicles,1Chr,1,4,4,None,1Chr_1_4,"Ной, Сим, Хам и Иафет."
4,russian_synodal_ru,Russian Synodal Bible (1876),Russian,13,1 Chronicles,1Chr,1,5,5,None,1Chr_1_5,"Сыновья Иафета: Гомер, Магог, Мадай, Иаван, Ел..."


In [3]:
df["chapter_original"] = df["chapter"]

# Восстанавливаем номера глав
df["chapter"] = (
    df.groupby("book_abbr", sort=False)["chapter"]
    .transform(lambda x: x.ne(x.shift()).cumsum())
)

# Пересоздаём идентификаторы стихов
df["verse_id"] = (
    df["book_abbr"]
    + "_" + df["chapter"].astype(str)
    + "_" + df["verse"].astype(str)
)

df['text'] = (df["chapter"].astype(str)
    + "_" + df["verse"].astype(str)
    + '_' + df['text']
)

In [4]:
data = df[['book', 'chapter', 'verse', 'text']]
data

,book,chapter,verse,text
0,1 Chronicles,1,1,"1_1_Адам, Сиф, Енос,"
1,1 Chronicles,1,2,"1_2_Каинан, Малелеил, Иаред,"
2,1 Chronicles,1,3,"1_3_Енох, Мафусал, Ламех,"
3,1 Chronicles,1,4,"1_4_Ной, Сим, Хам и Иафет."
4,1 Chronicles,1,5,"1_5_Сыновья Иафета: Гомер, Магог, Мадай, Иаван..."
...,...,...,...,...
31164,Zephaniah,3,16,3_16_В тот день скажут Иерусалиму: “не бойся” ...
31165,Zephaniah,3,17,"3_17_Господь Бог твой среди тебя, Он силен спа..."
31166,Zephaniah,3,18,3_18_Сетующих о торжественных празднествах Я с...
31167,Zephaniah,3,19,"3_19_Вот, Я стесню всех притеснителей твоих в ..."


In [5]:
# embedder = Embedder()
# vectors = embedder.encode(
#     [text for text in data['text']]
# )

In [6]:
tokenizer = Tokenizer()
def chuncker(data, max_length):
    chuncks = []
    chunck = {}
    for i in range(len(data)):
        if 'text' not in chunck:
            chunck['book'] = data.iloc[i]['book']
            chunck['chapter'] = data.iloc[i]['chapter'].item()
            chunck['text'] = data.iloc[i]['text']
            continue
        if len(tokenizer.encode(chunck['text'])) > max_length or chunck['book']!=data.iloc[i]['book'] or chunck['chapter']!=data.iloc[i]['chapter']:
            chuncks.append(chunck)
            chunck = {}
            continue
        chunck['text'] += data.iloc[i]['text']
    return chuncks

    
chuncks = chuncker(data, 250)
        
print(chuncks[:3])






[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'apply_yarn_scaling'}


[{'book': '1 Chronicles', 'chapter': 1, 'text': '1_1_Адам, Сиф, Енос,1_2_Каинан, Малелеил, Иаред,1_3_Енох, Мафусал, Ламех,1_4_Ной, Сим, Хам и Иафет.1_5_Сыновья Иафета: Гомер, Магог, Мадай, Иаван, Елиса, Фувал, Мешех и Фирас.1_6_Сыновья Гомера: Аскеназ, Рифат и Фогарма.1_7_Сыновья Иавана: Елиса, Фарсис, Киттим и Доданим.1_8_Сыновья Хама: Хуш, Мицраим, Фут и Ханаан.1_9_Сыновья Хуша: Сева, Хавила, Савта, Раама и Савтеха. Сыновья Раамы: Шева и Дедан.1_10_Хуш родил также Нимрода: сей начал быть сильным на земле.'}, {'book': '1 Chronicles', 'chapter': 1, 'text': '1_12_Патрусима, Каслухима, от которого произошли Филистимляне, и Кафторима.1_13_Ханаан родил Сидона, первенца своего, Хета,1_14_Иевусея, Аморрея, Гергесея,1_15_Евея, Аркея, Синея,1_16_Арвадея, Цемарея и Хамафея.1_17_Сыновья Сима: Елам, Ассур, Арфаксад, Луд и Арам. Сыновья Арама: Уц, Хул, Гефер и Мешех.1_18_Арфаксад родил Каинана, Каинан же родил Салу, Сала же родил Евера.1_19_У Евера родились два сына: имя одному Фалек, потому что в

In [8]:
from sentence_transformers import CrossEncoder
import time

model = CrossEncoder(
    "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
    device="cpu"
)

query = "Что Библия говорит о прощении?"

# Здесь должны быть 30 реальных чанков из Qdrant
pairs = [(query, chunk["text"]) for chunk in chunсks[:30]]

# Прогрев
model.predict(pairs[:2])

start = time.perf_counter()

scores = model.predict(
    pairs,
    batch_size=8
)

top5 = sorted(
    range(len(scores)),
    key=lambda i: scores[i],
    reverse=True
)[:5]

print(f"Время: {time.perf_counter() - start:.3f} сек")
print("Top-5:", top5)

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


NameError: name 'nn' is not defined